<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 9 — Machine Learning Triage
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_09.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 · 4 · 5 · 6 · 7 · **9 (this notebook)**

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | Isolation Forest anomaly scoring across the alerted account pool (mirrors Section 9.6 of the text) | — |
| **2. Exercise 9.1 Extension** | Feature importance analysis · contamination parameter sensitivity · governance framework | Exercise 9.1 |
| **3. Reflection cells** | Structured answer prompts | Exercise 9.1 Parts A–D |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist. All account IDs, names, and transactions are generated from a fixed random seed for educational purposes only.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: Isolation Forest ML Triage

> *This section mirrors Section 9.6 of the textbook exactly. Run the cells and compare the output to the printed figures.*

### The complete system architecture

By this chapter, the Northgate TM system has four components:

1. **Rule NRB-STRUCT-001** — detects sub-threshold cash structuring (Ch 4)
2. **Rule NRB-VEL-002** — detects rapid-fire velocity bursts (Ch 6)
3. **Rule NRB-GEO-003** — detects transactions with high-risk country counterparties (Ch 7)
4. **Isolation Forest triage layer** — scores all rule-triggered accounts by anomaly level and ranks the investigation queue (this chapter)

### What Isolation Forest does

Isolation Forest is an unsupervised anomaly detection algorithm. It "isolates" observations by randomly partitioning the feature space. Anomalous observations (those with unusual combinations of feature values) are isolated in fewer splits — they receive a lower (more negative) anomaly score. In our context, accounts exhibiting multiple extreme features simultaneously — high cash-in, high velocity, high geographic risk — will be isolated quickly and rank at the top of the ML-scored queue.

### The feature set

| Feature | Source |
|---------|--------|
| `total_cash_in` | Sum of all cash deposits in 2023 |
| `txn_count` | Total transactions in 2023 |
| `cash_ratio` | Fraction of transactions that are cash deposits |
| `hr_ratio` | Fraction of transactions involving high-risk country counterparties |
| `rule1_flag` | 1 if account triggered NRB-STRUCT-001, else 0 |
| `rule2_flag` | 1 if account triggered NRB-VEL-002, else 0 |
| `rule3_flag` | 1 if account triggered NRB-GEO-003, else 0 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

df_txn  = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
df_cpty = pd.read_csv('nb_counterparties.csv')
MULE_IDS = [f'ACC{i:04d}' for i in range(1, 7)]
HIGH_RISK_COUNTRIES = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']

# Rebuild all three rules
def apply_rule_1(df_txn, threshold=7500, min_txns=3):
    cash = df_txn[(df_txn['txn_type']=='CASH_IN') & (df_txn['amount']<10_000)].copy()
    cash = cash.sort_values(['account_id','txn_date'])
    results = []
    for acct, grp in cash.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rs = grp['amount'].rolling('30D').sum()
        rc = grp['amount'].rolling('30D').count()
        if rs.max() > threshold and rc.max() >= min_txns:
            results.append({'account_id': acct, 'peak_rolling_sum': round(rs.max(), 2)})
    return pd.DataFrame(results)

def apply_rule_2(df_txn, min_txns=5, window_days=14, max_gap_days=3):
    df = df_txn.copy().sort_values(['account_id','txn_date'])
    results = []
    for acct, grp in df.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rc = grp['amount'].rolling(f'{window_days}D').count()
        if rc.max() >= min_txns:
            gaps = grp.index.to_series().diff().dt.days.dropna()
            if (gaps <= max_gap_days).sum() >= (min_txns - 1):
                results.append({'account_id': acct})
    return pd.DataFrame(results)

def apply_rule_3(df_txn, df_cpty, high_risk_countries, min_txns=2, min_amount=5000):
    hr_cpty = df_cpty[df_cpty['country_code'].isin(high_risk_countries)]['counterparty_id']
    hr_txns = df_txn[df_txn['counterparty_id'].isin(hr_cpty)].copy()
    results = []
    for acct, grp in hr_txns.groupby('account_id'):
        if len(grp) >= min_txns and grp['amount'].sum() >= min_amount:
            results.append({'account_id': acct})
    return pd.DataFrame(results)

alerts_r1 = apply_rule_1(df_txn)
alerts_r2 = apply_rule_2(df_txn)
alerts_r3 = apply_rule_3(df_txn, df_cpty, HIGH_RISK_COUNTRIES)

all_alerted = set(alerts_r1['account_id']) | set(alerts_r2['account_id']) | set(alerts_r3['account_id'])
print(f"Alert pool — R1: {len(alerts_r1)}, R2: {len(alerts_r2)}, R3: {len(alerts_r3)}")
print(f"Union (unique accounts in alert pool): {len(all_alerted)}")

In [ ]:
# Build feature matrix for all alerted accounts
feats = df_txn.groupby('account_id').agg(
    total_cash_in=('amount', lambda x: x[df_txn.loc[x.index, 'txn_type'] == 'CASH_IN'].sum()),
    txn_count=('txn_id', 'count'),
    cash_ratio=('txn_type', lambda x: (x == 'CASH_IN').mean()),
).fillna(0)

hr_cpty_ids = set(df_cpty[df_cpty['country_code'].isin(HIGH_RISK_COUNTRIES)]['counterparty_id'])
hr_ratio = df_txn.groupby('account_id')['counterparty_id'].apply(
    lambda x: x.isin(hr_cpty_ids).mean()
)
feats['hr_ratio'] = hr_ratio.fillna(0)

# Rule flags
feats['rule1_flag'] = feats.index.isin(alerts_r1['account_id']).astype(int)
feats['rule2_flag'] = feats.index.isin(alerts_r2['account_id']).astype(int)
feats['rule3_flag'] = feats.index.isin(alerts_r3['account_id']).astype(int)

# Filter to alert pool only
alert_feats = feats[feats.index.isin(all_alerted)].copy()
feature_cols = ['total_cash_in', 'txn_count', 'cash_ratio', 'hr_ratio',
                'rule1_flag', 'rule2_flag', 'rule3_flag']
X = alert_feats[feature_cols].values

print(f"Feature matrix: {X.shape[0]} accounts × {X.shape[1]} features")
print()
print("Feature statistics:")
print(alert_feats[feature_cols].describe().round(2).to_string())

In [ ]:
# Fit Isolation Forest
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
alert_feats['if_score'] = iso.score_samples(X)   # lower = more anomalous

# Rank by anomaly score
ranked = alert_feats.sort_values('if_score').reset_index()
ranked['rank'] = ranked.index + 1
ranked['known_mule'] = ranked['account_id'].isin(MULE_IDS)

print("Top 15 accounts by Isolation Forest anomaly score (lower = more anomalous):")
print(ranked[['rank', 'account_id', 'total_cash_in', 'cash_ratio', 'hr_ratio',
              'rule1_flag', 'rule2_flag', 'rule3_flag', 'if_score', 'known_mule']]
      .head(15).to_string(index=False))
print()
print("Mule account ranks:")
print(ranked[ranked['known_mule']][['rank', 'account_id', 'if_score']].to_string(index=False))

In [ ]:
# Chart: IF score distribution with mule accounts highlighted
fig, ax = plt.subplots(figsize=(8, 4))
non_mule_scores = alert_feats[~alert_feats.index.isin(MULE_IDS)]['if_score']
mule_scores     = alert_feats[alert_feats.index.isin(MULE_IDS)]['if_score']

ax.hist(non_mule_scores, bins=30, color='#4472C4', alpha=0.7, label='Other alerted accounts')
for i, score in enumerate(sorted(mule_scores)):
    label = 'Mule accounts' if i == 0 else '_nolegend_'
    ax.axvline(score, color='#E74C3C', linewidth=2, alpha=0.9, label=label)

ax.set_xlabel('Isolation Forest Score (lower = more anomalous)', fontsize=10)
ax.set_ylabel('Number of Accounts', fontsize=10)
ax.set_title('Isolation Forest Anomaly Scores\nAmong All Rule-Triggered Accounts', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

**What you're seeing:** The mule accounts (red vertical lines) should cluster at the left tail of the distribution — the most anomalous scores. The Isolation Forest has no knowledge that these accounts are mules. It has discovered, purely from the feature values, that these accounts are unlike the rest of the alerted population.

This is the complete Northgate TM system: three rule-based filters generate an alert pool, and the Isolation Forest scores each alerted account to prioritise investigation effort. Investigators focus on the left tail; accounts in the middle of the distribution receive lower-priority review.

---
## Section 2 — Exercise 9.1 Extension: Model Governance

> *The main-text exercise asks you to run the Isolation Forest and inspect the ranked output. This extension asks you to test the model's sensitivity to the `contamination` parameter and begin building the governance documentation a model validator would expect.*

In [ ]:
# Contamination parameter sensitivity: how does the ranking change?
contamination_values = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]
sensitivity_results  = []

for cont in contamination_values:
    iso_c = IsolationForest(n_estimators=100, contamination=cont, random_state=42)
    scores_c = iso_c.score_samples(X)
    temp = alert_feats.copy()
    temp['if_score'] = scores_c
    ranked_c = temp.sort_values('if_score').reset_index()
    ranked_c['rank'] = ranked_c.index + 1
    mule_ranks = ranked_c[ranked_c['account_id'].isin(MULE_IDS)]['rank'].tolist()
    sensitivity_results.append({
        'contamination': cont,
        'mule_avg_rank': round(sum(mule_ranks)/len(mule_ranks), 1),
        'mule_max_rank': max(mule_ranks),
        'mule_min_rank': min(mule_ranks),
    })

sens_df = pd.DataFrame(sensitivity_results)
print("Isolation Forest — Contamination Parameter Sensitivity:")
print("(Mule account ranks — lower = better detection)")
print(sens_df.to_string(index=False))

**✏️ YOUR OBSERVATION**

Look at the sensitivity table. As `contamination` increases from 0.01 to 0.20:
- Does the average mule rank change significantly?
- Is the model robust (stable rankings) or sensitive (mule ranks jump around)?
- What contamination setting would you recommend and why?

*Write your answers in Section 3, Question 2.*

In [ ]:
# Model card: feature importance proxy via permutation (shuffle each feature, measure rank change)
import copy

baseline_mule_avg_rank = None
feature_importance = []

for col in feature_cols:
    X_perm = X.copy()
    col_idx = feature_cols.index(col)
    np.random.seed(42)
    X_perm[:, col_idx] = np.random.permutation(X_perm[:, col_idx])
    iso_p = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    scores_p = iso_p.score_samples(X_perm)
    temp = alert_feats.copy()
    temp['if_score'] = scores_p
    ranked_p = temp.sort_values('if_score').reset_index()
    ranked_p['rank'] = ranked_p.index + 1
    mule_ranks_p = ranked_p[ranked_p['account_id'].isin(MULE_IDS)]['rank'].tolist()
    avg_rank_p = sum(mule_ranks_p) / len(mule_ranks_p)
    if baseline_mule_avg_rank is None:
        iso_b = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
        iso_b.fit(X)
        scores_b = iso_b.score_samples(X)
        temp_b = alert_feats.copy()
        temp_b['if_score'] = scores_b
        ranked_b = temp_b.sort_values('if_score').reset_index()
        ranked_b['rank'] = ranked_b.index + 1
        baseline_mule_avg_rank = ranked_b[ranked_b['account_id'].isin(MULE_IDS)]['rank'].mean()
    feature_importance.append({'feature': col, 'avg_mule_rank_when_shuffled': round(avg_rank_p, 1)})

fi_df = pd.DataFrame(feature_importance).sort_values('avg_mule_rank_when_shuffled', ascending=False)
print(f"Baseline avg mule rank: {baseline_mule_avg_rank:.1f}")
print("\nPermutation importance (higher rank when shuffled = more important feature):")
print(fi_df.to_string(index=False))

**What you're seeing:** Permutation importance shuffles each feature independently and measures how much mule account ranks worsen as a result. Features that, when shuffled, push mule accounts further down the queue are the most important to the model's ability to detect them.

This is an approximation of feature importance — Isolation Forest does not have native feature importance like gradient boosting. But for model governance purposes, being able to say "the model's ranking is primarily driven by `hr_ratio` and `total_cash_in`" is essential documentation for a model validator.

---
## Section 3 — Reflection: Exercise 9.1 Answer Cells

> *Double-click any cell to edit it.*

#### Question 1 — Isolation Forest Results

*(Edit this cell to write your answer)*

**What were the ranks of the six mule accounts in the baseline Isolation Forest run?**  
  
**Were all six in the top 10? If not, which account ranked outside the top 10, and looking at its feature values, why might it have ranked lower?**  
  
**In plain English, explain to a non-technical compliance manager what the Isolation Forest score means and how it should influence investigation priority.**  


#### Question 2 — Contamination Parameter

*(Edit this cell to write your answer)*

**How stable are the mule account ranks as you vary the contamination parameter from 0.01 to 0.20?**  
  
**The `contamination` parameter represents the expected fraction of anomalies in the dataset. If your bank's compliance team estimates that 2% of alerted accounts are genuine suspicious activity, what contamination value would you set?**  
  
**Should the contamination parameter be disclosed to regulators as a model assumption? Why or why not?**  


#### Question 3 — Feature Importance and Model Governance

*(Edit this cell to write your answer)*

**Which two features have the highest permutation importance (i.e., shuffling them most damages mule recall)?**  
  
**If `rule1_flag` has low permutation importance, does that mean Rule NRB-STRUCT-001 is useless? Explain your reasoning.**  
  
**List three items that a model validator would expect to see in the model documentation for this Isolation Forest, based on SR 11-7 requirements.**  


#### Question 4 — System Reflection

*(Edit this cell to write your answer)*

**You have now built a complete four-layer TM system (three rules + ML triage). Describe in 3–4 sentences how the system works end to end — from raw transactions to a prioritised investigation queue.**  
  
**What is the biggest remaining weakness of the Northgate system as built? What would you add in a real deployment?**  
  
**The six mule accounts were embedded with structuring behaviour AND high-risk counterparties by design. In a real bank, what proportion of suspicious accounts do you think exhibit multiple red flags simultaneously?**  


---
## Section 4 — Capstone: Interactive Alert Investigation Dashboard

> *This section is the capstone of the Northgate TM system. Run all previous cells first — Section 4 depends on the `ranked` DataFrame and `alert_feats` built in Section 1.*

You have built a complete four-layer TM system. This section adds an interactive investigation interface powered by **ipywidgets** — the same library that underpins interactive controls in Jupyter and Colab. The dashboard lets you select any alerted account, see its full transaction history, rule flags, anomaly score, and write a structured disposition note — exactly as an investigator would in a real case management system.

> **Note for Google Colab users:** If the dropdown does not appear immediately, run the cell again or click **Runtime → Restart and run all**.

This is the complete Northgate Transaction Monitoring System as it stands at the end of Chapter 8:
- Three rule-based detection layers (NRB-STRUCT-001, NRB-VEL-002, NRB-GEO-003)
- One ML triage layer (Isolation Forest, seven features)
- One interactive investigation interface (this section)

In Chapter 9 you will see how a model validator assesses this system against the SR 11-7 / PRA CS 6/23 five-pillar framework.

In [ ]:
# ipywidgets is pre-installed in Colab — this cell just confirms the version
import ipywidgets as widgets
print(f"ipywidgets version: {widgets.__version__}")
print("✅ Ready — run the next cell to launch the investigation dashboard.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import matplotlib.pyplot as plt

# ── Rebuild required DataFrames if this cell is run standalone ────────────────
# (If you have already run Section 1, these are already in memory — skip.)
try:
    _ = ranked
except NameError:
    exec(open.__doc__)  # placeholder — in practice Section 1 must run first
    raise RuntimeError("Please run Section 1 before Section 4.")

# ── Build investigation data package ─────────────────────────────────────────
df_txn_raw  = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
df_cust_raw = pd.read_csv('nb_customers.csv')
df_cpty_raw = pd.read_csv('nb_counterparties.csv')

HIGH_RISK_COUNTRIES = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
hr_cpty_ids = set(df_cpty_raw[df_cpty_raw['country_code'].isin(HIGH_RISK_COUNTRIES)]['counterparty_id'])

# Build sorted account list: mule accounts first (they have lowest if_score), then rest
account_options = ranked['account_id'].tolist()

# ── Widgets ───────────────────────────────────────────────────────────────────
account_dd  = widgets.Dropdown(
    options=account_options,
    description='Account:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px')
)
prev_btn    = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
next_btn    = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
disposition = widgets.ToggleButtons(
    options=['Pending', 'Close — No Action', 'Escalate to L2', 'File SAR'],
    value='Pending',
    description='Disposition:',
    style={'description_width': '100px', 'button_width': '130px'},
)
notes_box   = widgets.Textarea(
    placeholder='Analyst notes — summarise your rationale here...',
    layout=widgets.Layout(width='98%', height='80px')
)
save_btn    = widgets.Button(
    description='💾  Save Disposition',
    button_style='success',
    layout=widgets.Layout(width='180px')
)
save_status = widgets.Label(value='')
out         = widgets.Output()

# Case log: stores dispositions across accounts
case_log = {}

# ── Render function ───────────────────────────────────────────────────────────
def render_case(account_id):
    with out:
        clear_output(wait=True)

        row = ranked[ranked['account_id'] == account_id].iloc[0]
        rank       = int(row['rank'])
        if_score   = float(row['if_score'])
        r1 = int(row['rule1_flag'])
        r2 = int(row['rule2_flag'])
        r3 = int(row['rule3_flag'])
        total_ci   = float(row['total_cash_in'])
        cash_r     = float(row['cash_ratio'])
        hr_r       = float(row['hr_ratio'])
        is_mule    = account_id in [f'ACC{i:04d}' for i in range(1, 7)]

        cust = df_cust_raw[df_cust_raw['account_id'] == account_id]
        txns = df_txn_raw[df_txn_raw['account_id'] == account_id].sort_values('txn_date')
        txns_hr = txns[txns['counterparty_id'].isin(hr_cpty_ids)]

        # ── Header ────────────────────────────────────────────────────────────
        mule_tag = "  ⚠️  [KNOWN TEST MULE]" if is_mule else ""
        print(f"{'='*70}")
        print(f"  ALERT INVESTIGATION — {account_id}{mule_tag}")
        print(f"  Isolation Forest rank: #{rank} of {len(ranked)}   |   Score: {if_score:.4f}")
        print(f"{'='*70}")

        # ── Customer profile ─────────────────────────────────────────────────
        if len(cust) > 0:
            c = cust.iloc[0]
            print(f"
  CUSTOMER PROFILE")
            print(f"  {'Occupation':<22} {c.get('occupation', 'N/A')}")
            print(f"  {'CRR Score':<22} {c.get('crr_score', 'N/A')}")
            print(f"  {'Stated Income (USD)':<22} {c.get('stated_income_usd', 'N/A'):,.0f}")
            print(f"  {'Account opened':<22} {c.get('account_open_date', 'N/A')}")

        # ── Rule flags ───────────────────────────────────────────────────────
        print(f"
  RULE FLAGS")
        print(f"  {'NRB-STRUCT-001 (Structuring)':<34} {'🔴 TRIGGERED' if r1 else '⬜ not triggered'}")
        print(f"  {'NRB-VEL-002 (Velocity)':<34} {'🔴 TRIGGERED' if r2 else '⬜ not triggered'}")
        print(f"  {'NRB-GEO-003 (Geo Risk)':<34} {'🔴 TRIGGERED' if r3 else '⬜ not triggered'}")
        print(f"  {'Rules triggered':<34} {r1+r2+r3} / 3")

        # ── Behavioural features ──────────────────────────────────────────────
        print(f"
  BEHAVIOURAL FEATURES (2023)")
        print(f"  {'Total cash in (USD)':<34} {total_ci:>12,.0f}")
        print(f"  {'Cash-in ratio':<34} {cash_r:>11.1%}")
        print(f"  {'High-risk counterparty ratio':<34} {hr_r:>11.1%}")
        print(f"  {'Total transactions':<34} {len(txns):>12,}")
        print(f"  {'Transactions to HR countries':<34} {len(txns_hr):>12,}")

        # ── Recent transactions ───────────────────────────────────────────────
        print(f"
  RECENT TRANSACTIONS (last 10)")
        recent = txns.tail(10)[['txn_date','txn_type','amount','counterparty_id']].copy()
        recent['hr'] = recent['counterparty_id'].isin(hr_cpty_ids).map({True:'⚠️ HR', False:''})
        recent['txn_date'] = recent['txn_date'].dt.strftime('%Y-%m-%d')
        recent['amount']   = recent['amount'].map('{:>10,.2f}'.format)
        for _, r in recent.iterrows():
            print(f"  {r['txn_date']}  {r['txn_type']:<14} USD {r['amount']}   {r['counterparty_id']}  {r['hr']}")

        # ── Monthly cash-in chart ─────────────────────────────────────────────
        cash_in = txns[txns['txn_type']=='CASH_IN'].copy()
        if len(cash_in) > 0:
            cash_in['month'] = cash_in['txn_date'].dt.to_period('M')
            monthly = cash_in.groupby('month')['amount'].sum()
            fig, ax = plt.subplots(figsize=(7, 2.8))
            ax.bar(monthly.index.astype(str), monthly.values, color='#4472C4', width=0.6)
            ax.axhline(7500, color='#E74C3C', linestyle='--', linewidth=1.2,
                       label='NRB-STRUCT-001 threshold (USD 7,500)')
            ax.set_title(f'Monthly Cash-In — {account_id}', fontsize=10, fontweight='bold')
            ax.set_ylabel('USD', fontsize=9)
            ax.tick_params(axis='x', rotation=45, labelsize=8)
            ax.legend(fontsize=8)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.tight_layout()
            plt.show()

        # ── Previous disposition ─────────────────────────────────────────────
        if account_id in case_log:
            prev = case_log[account_id]
            print(f"
  PREVIOUS DISPOSITION: {prev['disposition']}")
            print(f"  Notes: {prev['notes']}")
        print()

# ── Button callbacks ─────────────────────────────────────────────────────────
def on_prev(b):
    idx = account_options.index(account_dd.value)
    if idx > 0:
        account_dd.value = account_options[idx - 1]

def on_next(b):
    idx = account_options.index(account_dd.value)
    if idx < len(account_options) - 1:
        account_dd.value = account_options[idx + 1]

def on_save(b):
    acct = account_dd.value
    case_log[acct] = {'disposition': disposition.value, 'notes': notes_box.value}
    save_status.value = f"✅ Saved: {acct} → {disposition.value}"

def on_account_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        acct = change['new']
        # Restore previous disposition if exists
        if acct in case_log:
            disposition.value = case_log[acct]['disposition']
            notes_box.value   = case_log[acct]['notes']
        else:
            disposition.value = 'Pending'
            notes_box.value   = ''
        save_status.value = ''
        render_case(acct)

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save)
account_dd.observe(on_account_change)

# ── Layout ───────────────────────────────────────────────────────────────────
nav_bar     = widgets.HBox([account_dd, prev_btn, next_btn,
                             widgets.Label('   (sorted by anomaly rank: most suspicious first)')])
action_bar  = widgets.HBox([save_btn, save_status])
panel       = widgets.VBox([
    nav_bar,
    out,
    widgets.HTML('<b>Analyst Notes</b>'),
    notes_box,
    disposition,
    action_bar,
], layout=widgets.Layout(border='1px solid #cccccc', padding='10px', border_radius='6px'))

display(panel)
render_case(account_options[0])

---
### ✏️ Investigation Notes — Structured Reflection

After working through the investigation dashboard, use the cell below to record your observations. Consider:

1. **Did all six mule accounts appear in the top 6 by anomaly rank?** If any appeared lower, what feature values explain the lower rank?

2. **Pick one non-mule account that appeared in the top 20.** What is its anomaly score? Which features drove its ranking? Is this a credible false positive or does the account show genuinely unusual behaviour?

3. **In a real bank, what additional information would you want in this investigation view** that the Northgate dataset does not provide (e.g., KYC documents, account opening notes, previous SAR history)?

4. **The disposition buttons (Close / Escalate / File SAR) represent a simplified L1 review outcome.** In a real TMS, what additional approval steps would be required before a SAR could be filed?

*Write your answers in the markdown cell below.*

*(Edit this cell to record your investigation observations)*

**Top 6 by anomaly rank — mule account coverage:**  

**Non-mule account in top 20 — analysis:**  

**Additional information needed in a real investigation view:**  

**SAR filing approval steps in a real TMS:**  


---
## Congratulations — the Northgate TM System is complete

You have built, from scratch, a working four-layer transaction monitoring system:

| Layer | Component | Chapter |
|-------|-----------|---------|
| 1 | Rule NRB-STRUCT-001 — Cash Deposit Structuring | 4 |
| 2 | Customer Segmentation — K-Means Peer Groups | 5 |
| 3 | Rules NRB-VEL-002 and NRB-GEO-003 — Velocity and Geo Risk | 6–7 |
| 4 | Isolation Forest ML Triage — Anomaly Scoring and Ranking | 8 |
| 5 | Interactive Investigation Dashboard — ipywidgets UI | 8 (this section) |

In **Chapter 9** you will step into the role of the model validator and assess the full system against the SR 11-7 / PRA CS 6/23 five-pillar model risk management framework: conceptual soundness, data integrity, performance testing, sensitivity analysis, and ongoing monitoring.

---
*© Compliance Analytics Ltd. All dataset content is synthetic and fictional.*